# 次のステップ整理：木材近赤外スペクトルによる含水率予測

作成: 2026-04-17  
目的: 現状を踏まえ、何をどの順番で試すべきかを整理する

---

## 現状スコアまとめ

| バージョン | 前処理 | 特徴量 | モデル | CV方式 | 提出RMSE |
|---|---|---|---|---|---|
| v1 | SNV + PCA(50) | PCA50 + species番号 | LightGBM | sample単位（リーク有） | 20.062 |
| **v2** | SNV + PCA(50) | PCA50 + species番号 | LightGBM | **樹種単位（正しい）** | **18.092** |
| v3 | SNV + SG2次微分 | 生スペクトル全波数 | PLSR | 樹種単位 | 23.037 |

→ 現時点のベストは **v2 (RMSE=18.092)**

---

## 最重要課題：ゼロショット汎化

**trainとtestで樹種が完全に異なる（共通樹種ゼロ）**

- train: イチョウ・ウエンジ・ウォールナット・クリ・スプルース・チェリー・トチ・ナラ・ヒノキ・ベイスギ・ベイマツ・ホワイトオーク・米ヒバ（13種）
- test: クスノキ・ケヤキ・スギ・タモ・チーク・ヤマザクラ（6種）

**これが本コンペの本質的な難しさ。**  
「樹種固有の特性」ではなく「水の吸収」というドメイン知識を活かした特徴量・モデル設計が必要。

### CV設計の正解
- `GroupKFold(groups=樹種)` で test-like な評価をする
- sample単位の分割はリークになるので NG
- ベイスギはCVのfoldから除外（汎化を妨げる可能性。v2で検証済み）

---

## 前処理方針

### 現在の実装
```
SNV → PCA(50次元)  ←  LightGBM に投入
SNV → SG2次微分    ←  PLSR に投入
```

### SNV（Standard Normal Variate）
- **効果**: 各スペクトルを行単位で標準化（平均0・分散1）。散乱・光路長変動を補正
- **判断**: 必須。引き続き全手法で使う
- 改善できることは → 特になし（SNVは理論的に正しい）

### SG 2次微分（Savitzky-Golay）
- **効果**: ベースラインのオフセット・傾きを除去。水ピークがより際立つ
- **現状**: window=11, poly=2, deriv=2
- 改善できることは → **windowサイズのチューニング**（5, 7, 15, 21を試す）  
  - windowが小さい → ノイズを拾いやすい  
  - windowが大きい → 微分が鈍くなる  
  - CV RMSEで最適値を選ぶ

### MSC（Multiplicative Scatter Correction）【未実装・要試験】
- **効果**: trainの平均スペクトルを基準に散乱成分を補正。SNVより散乱特異的
- **注意**: trainの平均スペクトルをfitして、testに1サンプルずつ適用する形なら**ルール上OK**
- 改善できることは → SNV単独 vs MSC単独 vs SNV+MSC をCVで比較する

### 波数帯域の絞り込み【重要・未実施】
- NIRで含水率に直接関与するのは水の吸収帯
  - **~8500 cm⁻¹**: OHの第2倍音
  - **~6900 cm⁻¹**: OHの第1倍音  
  - **~5200 cm⁻¹**: OHの結合音
- 全1555波数をそのまま使うと不要なノイズも学習してしまう
- 改善できることは → **水ピーク周辺±200 cm⁻¹程度に限定**して学習（特に樹種間で安定した帯域を優先）

---

## 特徴量エンジニアリング方針

### 現在の特徴量
- PCA50次元 + species番号（LightGBM用）
- 生スペクトル全波数（PLSR用）

### species番号の問題
- train: 1〜13、test: 14〜19（？）→ LightGBMには未知カテゴリ
- **species番号は使わないか、embedding化が必要**

### 改善案① 水ピーク帯域から統計量を手作り
```python
# 各ピーク帯域の面積・最大値・ピーク位置などを特徴量化
peak_regions = {
    '8500': (8300, 8700),
    '6900': (6700, 7100),
    '5200': (5000, 5400),
}
```
- 改善できることは → PCA50より物理的に意味のある低次元表現になる可能性

### 改善案② 樹種名のテキストembedding化【ルール上許可】
- `sentence-transformers` の多言語モデルで樹種名（日本語）をベクトル化
- `パラメータを更新しない（重みを変えずにembeddingを取得）` ならルールOK
- trainの13樹種のembedding→含水率の関係を学習し、test6樹種に汎化させる
- 改善できることは → 樹種の意味的近さをモデルに教えられる（例: スギとヒノキは近い）

### 改善案③ PCA次元数の見直し
- 現在: 50次元。寄与率を確認して最適値を探す
- 改善できることは → 30〜100の範囲でCVチューニング

---

## モデル方針

### 現在のモデル
- **LightGBM** (v1/v2): 汎用的な勾配ブースティング。PCA50を入力
- **PLSR** (v3): スペクトル解析の王道。全波数を直接入力

### LightGBM
- 改善できることは →
  - `num_leaves`, `learning_rate`, `min_child_samples` のチューニング（optuna推奨）
  - species番号を除外、または embedding特徴量に置き換え
  - 波数帯域絞り込み後のPCAを入力にする

### PLSR（Partial Least Squares Regression）
- スペクトル → 含水率の直接モデリングに強い
- v3のスコア(23.037)が悪かった原因: SG+PLSRの組み合わせ、またはn_componentsの設定
- 改善できることは →
  - **SNV+PCAなしの生スペクトルをPLSRに直接投入**して試す
  - n_componentsをCVで再チューニング（現在のCVが正しく機能しているか要確認）
  - SG微分ありとなしで比較

### 新規モデル候補①: Ridge / ElasticNet
- スペクトルデータ（高次元・共線性あり）に強い
- PLSRより実装が単純で解釈しやすい
- 改善できることは → SNV後の全波数に直接Ridgeをかける（PCA不要）

### 新規モデル候補②: SVR（Support Vector Regression）
- 少ないサンプルでも機能しやすい
- kernel='rbf' か kernel='linear' を試す

### アンサンブル戦略【最優先で試す価値あり】
```
最終予測 = α × LightGBM予測 + (1-α) × PLSR予測
```
- v2(LGBM=18.092)とv3(PLSR=23.037)は手法が全く異なるため、誤差が相補的な可能性
- 改善できることは → αをCVで最適化（0.1刻みで探索）
- まずα=0.7（LGBM重め）から試す

---

## 優先実施リスト（上から順に試す）

### 🔴 即効性高・実装コスト低

1. **PLSRの再チューニング**  
   - SNV後の生スペクトルをそのままPLSRに入れてCV評価
   - n_componentsを5〜30で探索  
   - → v3が23.037だった原因を特定する

2. **LightGBM + PLSRのアンサンブル**  
   - v2の予測とPLSR(再チューニング後)の予測を混ぜる  
   - `pred = 0.7 * pred_lgbm + 0.3 * pred_plsr` から試す

3. **波数帯域の絞り込み**  
   - 8300〜8700, 6700〜7100, 5000〜5400 cm⁻¹ の3帯域に限定  
   - SNV後にこの帯域だけ抽出してPLSR/LightGBMに投入

4. **SGのwindowチューニング**  
   - window = [5, 7, 11, 15, 21] でCVを回してベストを選ぶ

### 🟡 効果不明・実装コスト中

5. **MSCの追加**  
   - trainの平均スペクトルでfitし、train/testそれぞれ変換  
   - SNV+MSCとSNV単独を比較

6. **Ridge/ElasticNet**  
   - SNV後の全波数に対して直接学習（PCA不要）  
   - alphaをCV最適化

7. **ベイスギ以外の除外候補探索**  
   - 樹種別OOF残差を確認し、外れ値的な樹種を特定  
   - 除外してCVが改善するか確認

### 🟢 高難度・高リターン（余裕があれば）

8. **樹種名のテキストembedding化**  
   - `sentence-transformers` の多言語モデルを使用  
   - 樹種embedding + スペクトルPCAをLightGBMに投入  
   - testの6樹種に汎化できるか確認

---

## 実装テンプレート

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter

# --- データ読み込み ---
train = pd.read_csv("../data/raw/train.csv", encoding="shift-jis")
test  = pd.read_csv("../data/raw/test.csv",  encoding="shift-jis")

META_COLS   = ["sample number", "species number", "樹種", "含水率"]
wave_cols   = [c for c in train.columns if c not in META_COLS]
wavenumbers = np.array(wave_cols, dtype=float)

X_train_raw = train[wave_cols].values.astype(float)
y_train     = train["含水率"].values
groups      = train["樹種"].values

X_test_raw  = test[wave_cols].values.astype(float)

print(f"train shape: {X_train_raw.shape}, test shape: {X_test_raw.shape}")

In [ ]:
# --- 前処理関数 ---

def snv(X):
    """Standard Normal Variate: 行単位の標準化"""
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / (std + 1e-8)

def sg_derivative(X, window=11, poly=2, deriv=2):
    """Savitzky-Golay 2次微分"""
    return np.apply_along_axis(
        lambda x: savgol_filter(x, window_length=window, polyorder=poly, deriv=deriv),
        axis=1, arr=X
    )

def msc(X_train, X_new):
    """
    MSC: trainの平均スペクトルを基準に補正
    X_new は1サンプルでも機能する（ルール上OK）
    """
    ref = X_train.mean(axis=0)
    result = np.zeros_like(X_new)
    for i in range(len(X_new)):
        coef = np.polyfit(ref, X_new[i], 1)
        result[i] = (X_new[i] - coef[1]) / coef[0]
    return result, ref

def select_water_bands(X, wavenumbers):
    """水の吸収帯のみを抽出"""
    bands = [
        (8300, 8700),  # OHの第2倍音
        (6700, 7100),  # OHの第1倍音
        (5000, 5400),  # OHの結合音
    ]
    mask = np.zeros(len(wavenumbers), dtype=bool)
    for lo, hi in bands:
        mask |= ((wavenumbers >= lo) & (wavenumbers <= hi))
    return X[:, mask], wavenumbers[mask]

print("前処理関数を定義しました")

In [ ]:
# --- CV評価の雛形（GroupKFold・樹種単位）---

def cv_evaluate(X, y, groups, model_fn, n_splits=5, exclude_species=None):
    """
    Parameters
    ----------
    X : np.ndarray  前処理済み特徴量
    y : np.ndarray  目的変数
    groups : np.ndarray  樹種ラベル（GroupKFold用）
    model_fn : callable  モデルを返す関数（引数なし）
    exclude_species : list  CVから除外する樹種名
    """
    if exclude_species:
        mask = ~np.isin(groups, exclude_species)
        X, y, groups = X[mask], y[mask], groups[mask]

    gkf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(y))
    fold_rmses = []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model = model_fn()
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val).ravel()

        oof_preds[val_idx] = pred
        rmse = mean_squared_error(y_val, pred, squared=False)
        fold_rmses.append(rmse)

        val_species = np.unique(groups[val_idx])
        print(f"  Fold {fold+1}  val_species={val_species}  RMSE={rmse:.4f}")

    oof_rmse = mean_squared_error(y, oof_preds, squared=False)
    print(f"\n  OOF RMSE: {oof_rmse:.4f}")
    return oof_preds, oof_rmse, fold_rmses

print("CV評価関数を定義しました")

In [ ]:
# --- 優先実施 #1: PLSRの再チューニング ---
# SNV後の生スペクトルをそのままPLSRに投入

X_snv = snv(X_train_raw)

best_n, best_rmse = None, 1e9
results = []

for n_comp in [5, 8, 10, 15, 20, 25, 30]:
    print(f"\n--- n_components={n_comp} ---")
    _, rmse, _ = cv_evaluate(
        X_snv, y_train, groups,
        model_fn=lambda nc=n_comp: PLSRegression(n_components=nc),
        n_splits=5,
        exclude_species=["ベイスギ"]
    )
    results.append((n_comp, rmse))
    if rmse < best_rmse:
        best_rmse, best_n = rmse, n_comp

print(f"\nベスト n_components={best_n}  OOF RMSE={best_rmse:.4f}")

In [ ]:
# --- 優先実施 #2: アンサンブル ---
# v2のLightGBM予測値（submission_lgbm_v2.csv）とPLSR予測をブレンド
# ここでは仮の変数として示す

# pred_lgbm = ...  # v2のtest予測値（shape: (n_test,)）
# pred_plsr = ...  # PLSR再チューニング後のtest予測値

# for alpha in np.arange(0.5, 1.0, 0.1):
#     pred_ensemble = alpha * pred_lgbm + (1 - alpha) * pred_plsr
#     # CVで評価 → 最適αを選んでsubmit
#     print(f"alpha={alpha:.1f}  ...")

print("アンサンブルのテンプレートを確認しました")

In [ ]:
# --- 優先実施 #3: 水ピーク帯域絞り込み ---

X_snv_all = snv(X_train_raw)
X_water, wave_water = select_water_bands(X_snv_all, wavenumbers)
print(f"水ピーク帯域に絞った波数数: {X_water.shape[1]}")

# PLSRで評価
print("\n--- 水ピーク帯域 × PLSR ---")
_, rmse_water, _ = cv_evaluate(
    X_water, y_train, groups,
    model_fn=lambda: PLSRegression(n_components=10),
    n_splits=5,
    exclude_species=["ベイスギ"]
)

In [ ]:
# --- 優先実施 #4: SGのwindowチューニング ---

X_snv_all = snv(X_train_raw)

for window in [5, 7, 11, 15, 21]:
    try:
        X_sg = sg_derivative(X_snv_all, window=window, poly=2, deriv=2)
        print(f"\n--- SG window={window} ---")
        _, rmse, _ = cv_evaluate(
            X_sg, y_train, groups,
            model_fn=lambda: PLSRegression(n_components=10),
            n_splits=5,
            exclude_species=["ベイスギ"]
        )
    except Exception as e:
        print(f"window={window}: エラー ({e})")

In [ ]:
# --- submission生成テンプレート ---

def make_submission(test_df, pred, out_path):
    sub = test_df[["sample number"]].copy()
    sub["含水率"] = pred
    sub.to_csv(out_path, index=False, header=False, encoding="shift-jis")
    print(f"submission saved: {out_path}")
    return sub

# 使用例
# make_submission(test, pred_final, "../data/processed/submission_v4.csv")